In [11]:
#Import pandas
import pandas as pd

In [ ]:
#Read the CSV from the data folder
df = pd.read_csv("data/raw/hospitalization_discharge.csv")



Rows and columns: (2008, 21)
<class 'pandas.DataFrame'>
RangeIndex: 2008 entries, 0 to 2007
Data columns (total 21 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   inpatient_number                                2008 non-null   int64  
 1   destinationdischarge                            2008 non-null   str    
 2   admission_ward                                  2008 non-null   str    
 3   admission_way                                   2008 non-null   str    
 4   discharge_department                            2008 non-null   str    
 5   visit_times                                     2008 non-null   int64  
 6   respiratory_support                             42 non-null     str    
 7   oxygen_inhalation                               2008 non-null   str    
 8   dischargeday                                    2008 non-null   int64  
 9   admission_date         

['inpatient_number',
 'destinationdischarge',
 'admission_ward',
 'admission_way',
 'discharge_department',
 'visit_times',
 'respiratory_support',
 'oxygen_inhalation',
 'dischargeday',
 'admission_date',
 'outcome_during_hospitalization',
 'death_within_28_days',
 're_admission_within_28_days',
 'death_within_3_months',
 're_admission_within_3_months',
 'death_within_6_months',
 're_admission_within_6_months',
 'time_of_death__days_from_admission',
 'readmission_time_days_from_admission',
 'return_to_emergency_department_within_6_months',
 'time_to_emergency_department_within_6_months']

1. Renamed inpatient_number to patient_id.

Reasoning:
inpatient_number is essentially the unique identifier for each hospitalization/patient record. Renaming it to PatientID makes the dataset easier to understand and use in analysis, dashboards, and joins with other datasets.

In [ ]:
df.rename(columns={'inpatient_number': 'patient_id'}, inplace=True)

2. Renamed destinationdischarge to destination_discharge.

Reasoning:
The column describes where the patient went after discharge, such as Home, Healthcare Facility, or Died. Using a clearer name with an underscore improves readability and follows a consistent naming convention.

In [34]:
df.rename(columns={'destinationdischarge': 'destination_discharge'}, inplace=True)

3. Handled the blank values in respiratory_support.

Reasoning:
There are 1,966 blank values in this column. The non-blank values are IMV and NIMV.

Because a blank does not necessarily mean that respiratory information is missing—it can reasonably indicate that the patient did not receive IMV or NIMV—I recommend replacing the blanks with None rather than dropping the records.

This also makes the data easier to analyze because you can distinguish:

IMV → Invasive Mechanical Ventilation
NIMV → Non-Invasive Mechanical Ventilation
None → No IMV/NIMV recorded
Why I prefer None instead of deleting the rows:
There are 1,966 affected records, so dropping them would remove almost the entire dataset for this field and could introduce significant bias.

In [35]:
df['respiratory_support'] = df['respiratory_support'].fillna('None')

4. Renamed dischargeday to discharge_day.

Reasoning:
The existing column name is difficult to read because the words are combined. discharge_day clearly indicates that the value represents the number of days until discharge.

In [36]:
df.rename(columns={'dischargeday': 'discharge_day'}, inplace=True)

5. Verified and standardized admission_date.

Reasoning:
The admission_date values are currently stored as text/object values. All 2,008 records can be successfully interpreted as dates, so there are no invalid date values in this column.

Converting it to a proper datetime format will allow you to easily perform:

Year/month analysis
Admission trends
Time-based filtering
Sorting
Date calculations

In [37]:
df['admission_date'] = pd.to_datetime(
    df['admission_date'],
    errors='coerce'
)


6. Handled the blank values in time_of_death__days_from_admission.

Reasoning:
There are 1,964 blank values in this column. This field represents the number of days from admission until death.

A blank value is meaningful here because most patients did not have a recorded death, so we should not replace the blanks with 0. Replacing blanks with 0 would incorrectly suggest that the patient died on the day of admission.
Created a separate column for easier analysis death_recorded with 1 being Yes and 0 being No

In [38]:
# Keep blanks as NaN because they indicate no recorded time of death
df['time_of_death__days_from_admission'] = pd.to_numeric(
    df['time_of_death__days_from_admission'],
    errors='coerce'
)
df['death_recorded'] = df['time_of_death__days_from_admission'].notna().astype(int)

7. Handled the 1 blank in return_to_emergency_department_within_6_months.

Reasoning:
This is a binary field containing 0 and 1, with only one missing value.

For the one missing record, there isn't enough information in the related fields to confidently determine whether the patient returned to the emergency department. Therefore, I would not automatically convert it to 0, because that would make an assumption about the patient's outcome.

For a clean analytical dataset, we can label the missing value as Unknown.
This results in:

0 → No return to emergency department
1 → Returned to emergency department
Unknown → Information not available

This is safer than filling the blank with 0, because 0 represents an actual outcome, whereas the blank represents missing information.

In [39]:
df['return_to_emergency_department_within_6_months'] = (
    df['return_to_emergency_department_within_6_months']
    .fillna('Unknown')
)

In [44]:
# save the cleaned file
df.to_csv(
    "data/cleaned/hospitalization_discharge_cleaned.csv",
    index=False
)
import os

print(os.listdir("data/cleaned"))
cleaned_df = pd.read_csv(
    "data/cleaned/hospitalization_discharge_cleaned.csv"
)
df.columns.tolist()
cleaned_df.head()

['hospitalization_discharge_cleaned.csv']


,PatientID,Destination_discharge,admission_ward,admission_way,discharge_department,visit_times,respiratory_support,oxygen_inhalation,discharge_day,admission_date,...,re_admission_within_28_days,death_within_3_months,re_admission_within_3_months,death_within_6_months,re_admission_within_6_months,time_of_death__days_from_admission,readmission_time_days_from_admission,return_to_emergency_department_within_6_months,time_to_emergency_department_within_6_months,death_recorded
0,857781,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,11,2017-01-24,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0
1,743087,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,8,2017-05-05,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0
2,866418,Home,Cardiology,NonEmergency,Cardiology,2,NaN,OxygenTherapy,5,2016-11-18,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0
3,775928,Home,Cardiology,Emergency,Cardiology,1,NaN,OxygenTherapy,11,2017-10-02,...,1,0,1,0,1,NaN,19.0,1.0,19.0,0
4,810128,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,5,2019-11-17,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0


In [31]:
# save the cleaned file
df.to_csv(
    "data/cleaned/hospitalization_discharge_cleaned.csv",
    index=False
)
import os

print(os.listdir("data/cleaned"))
cleaned_df = pd.read_csv(
    "data/cleaned/hospitalization_discharge_cleaned.csv"
)

cleaned_df.head()

['hospitalization_discharge_cleaned.csv']


,PatientID,Destination_discharge,admission_ward,admission_way,discharge_department,visit_times,respiratory_support,oxygen_inhalation,discharge_day,admission_date,...,re_admission_within_28_days,death_within_3_months,re_admission_within_3_months,death_within_6_months,re_admission_within_6_months,time_of_death__days_from_admission,readmission_time_days_from_admission,return_to_emergency_department_within_6_months,time_to_emergency_department_within_6_months,death_recorded
0,857781,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,11,2017-01-24,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0
1,743087,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,8,2017-05-05,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0
2,866418,Home,Cardiology,NonEmergency,Cardiology,2,NaN,OxygenTherapy,5,2016-11-18,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0
3,775928,Home,Cardiology,Emergency,Cardiology,1,NaN,OxygenTherapy,11,2017-10-02,...,1,0,1,0,1,NaN,19.0,1.0,19.0,0
4,810128,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,5,2019-11-17,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0


In [32]:
# save the cleaned file
df.to_csv(
    "data/cleaned/hospitalization_discharge_cleaned.csv",
    index=False
)
import os

print(os.listdir("data/cleaned"))
cleaned_df = pd.read_csv(
    "data/cleaned/hospitalization_discharge_cleaned.csv"
)

cleaned_df.head()

['hospitalization_discharge_cleaned.csv']


,PatientID,Destination_discharge,admission_ward,admission_way,discharge_department,visit_times,respiratory_support,oxygen_inhalation,discharge_day,admission_date,...,re_admission_within_28_days,death_within_3_months,re_admission_within_3_months,death_within_6_months,re_admission_within_6_months,time_of_death__days_from_admission,readmission_time_days_from_admission,return_to_emergency_department_within_6_months,time_to_emergency_department_within_6_months,death_recorded
0,857781,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,11,2017-01-24,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0
1,743087,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,8,2017-05-05,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0
2,866418,Home,Cardiology,NonEmergency,Cardiology,2,NaN,OxygenTherapy,5,2016-11-18,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0
3,775928,Home,Cardiology,Emergency,Cardiology,1,NaN,OxygenTherapy,11,2017-10-02,...,1,0,1,0,1,NaN,19.0,1.0,19.0,0
4,810128,Home,Cardiology,NonEmergency,Cardiology,1,NaN,OxygenTherapy,5,2019-11-17,...,0,0,0,0,0,NaN,NaN,0.0,NaN,0
